# Bài tập về nhà  
Giao diện cho bài toán máy hút bụi với các thuật toán tìm kiếm:  

**Buổi 10:**  
Cost tính bằng h(n), cài đặt bằng khoảng cách Manhattan từ robot đến bụi GẦN NHẤT + số ô còn bụi  
- [Local Beam Search](#local-beam-search)  
    k = 2  
- [Simulated Annealing (Mô phỏng luyện kim)](#simulated-annealing)  
    T0 = 100  
    T_min = 0.01  
    alpha = 0.99  



Ở buổi 5 đã xây dựng các thuật toán:
- BFS cách tiếp cận 1  
- BFS cách tiếp cận 2  
- DFS cách tiếp cận 1  
- DFS cách tiếp cận 2  

Ở buổi 6 đã xây dựng các thuật toán:  
- IDS cách tiếp cận 1  
- IDS cách tiếp cận 2  
- UCS  

Ở buổi 7 đã xây dựng các thuật toán:  
- Greedy Search  
- A*  
  
Ở buổi 8 & 9 đã cây dựng các thuật toán:  
- IDA*  
- Leo đồi:  
    + Leo đồi đơn giản  
    + Leo đồi dốc nhất  
    + Leo đồi ngẫu nhiên
    + Leo đồi khởi tạo ngẫu nhiên   

**Link github:**  

**BT cá nhân:** Mỗi thuật toán tổ chức thành các module, file gui.ipynb sẽ import các thuật toán và gọi nó khi giao diện chạy thuật toán.  
https://github.com/Duc-Luong060106/vacuum_cleaner_personal_project  

In [10]:
# Class định nghĩa các thông tin của trạng thái
class Node:
    def __init__(self, state, parent, action, path_cost):
        self.state = state
        self.parent = parent
        self.action = action
        self.path_cost = path_cost

In [11]:
# Trả về list là chuỗi đường đi đến node truyền vào
def get_solution(node):
    path = []

    while node != None:
        path.append(node)
        node = node.parent

    path.reverse()
    return path


def print_solution(result):
    if result == None:
        print("Không tìm thấy đường đi!!!")
    else:
        for idx, node in enumerate(result):
            print(f"Bước {idx}: {node.action if node.action != None else "Khởi tạo"}")
            for row in node.state:
                print(*row)
            
            print()

        print("Phòng đã được dọn dẹp!!!")

In [12]:
# So sánh trạng thái với trạng thái đích (Không còn bụi)
def compare_state(state, goal_state):
    for x in range(len(state)):
        for y in range(len(state[0])):
            # Vị trí của robot đang đứng đã hút sạch bụi
            if state[x][y] != 'X' and state[x][y] != goal_state[x][y]:
                return False

    return True

In [13]:
def find_vacuum_position(state):
    for x in range(len(state)):
        for y in range(len(state[0])):

            if state[x][y] == 'X':
                return x, y
    
    return None

# Tạo ra các trạng thái con hợp lệ và hành động để sinh ra các trạng thái đó
def gen_actions(state):
    m = len(state)
    n = len(state[0])

    x, y = find_vacuum_position(state)

    actions = []

    if x > 0 and state[x-1][y] != 2:
        up_state = [row[:] for row in state]
        clean_action = f" và dọn rác ô [{x-1}][{y}]" if up_state[x-1][y] == 1 else ""

        up_state[x][y], up_state[x-1][y] = 0, 'X'

        actions.append(("Up" + clean_action, up_state))

    if x < m - 1 and state[x+1][y] != 2:
        down_state = [row[:] for row in state]
        clean_action = f" và dọn rác ô [{x+1}][{y}]" if down_state[x+1][y] == 1 else ""

        down_state[x][y], down_state[x+1][y] = 0, 'X'

        actions.append(("Down" + clean_action, down_state))

    if y > 0 and state[x][y-1] != 2:
        left_state = [row[:] for row in state]
        clean_action = f" và dọn rác ô [{x}][{y-1}]" if left_state[x][y-1] == 1 else ""

        left_state[x][y], left_state[x][y-1] = 0, 'X'

        actions.append(("Left" + clean_action, left_state))

    if y < n - 1 and state[x][y+1] != 2:
        right_state = [row[:] for row in state]
        clean_action = f" và dọn rác ô [{x}][{y+1}]" if right_state[x][y+1] == 1 else ""

        right_state[x][y], right_state[x][y+1] = 0, 'X'

        actions.append(("Right" + clean_action, right_state))

    return actions

In [14]:
# Nhóm thuật toán này sử dụng chi phí heuristic (h(n)), được cài đặt là khoảng cách heuristic GẦN NHẤT từ robot đến rác + số ô còn rác.
def calculate_heuristic_cost(state):
    x, y = find_vacuum_position(state)
    manhattan_cost = float('inf')
    dirty_cell = 0

    for i in range(len(state)):
        for j in range(len(state[0])):
            if state[i][j] == 1:
                dirty_cell += 1
                manhattan_distance = abs(x-i) + abs(y-j)
                manhattan_cost = min(manhattan_cost, manhattan_distance)
    
    return manhattan_cost + dirty_cell

In [15]:
def state_to_tuple(state):
    return tuple(tuple(row) for row in state)

## Local Beam Search

In [33]:
import random
import heapq

def local_beam_search(initial, goal, k):
    start = Node(initial, None, None, calculate_heuristic_cost(initial))

    if compare_state(start.state, goal):
        return get_solution(start)
    
    states_from_start = gen_actions(start.state)
    k_start_states = random.sample(states_from_start, min(k, len(states_from_start)))   # Sinh k trạng thái ban đầu

    current_state_set = []

    for action, state in k_start_states:
        node = Node(state, start, action, calculate_heuristic_cost(state))
        current_state_set.append(node)

    visited = set()     # Tránh lặp vô hạn khi đã duyệt qua tập current đó rồi

    while True:
        current_beam_signature = frozenset(state_to_tuple(node.state) for node in current_state_set)
        if current_beam_signature in visited:
            break       # Nếu bộ k state đó đã xét rồi thì dừng
            
        visited.add(current_beam_signature)
        neighbor_states = dict()

        for node in current_state_set:
            for action, child_state in gen_actions(node.state):

                tuple_state = state_to_tuple(child_state)
                if tuple_state not in neighbor_states:
                    
                    neighbor_states[tuple_state] = Node(child_state, node, action, calculate_heuristic_cost(child_state))

                    if compare_state(child_state, goal):
                        return get_solution(neighbor_states[tuple_state])

        k_best_nodes = heapq.nsmallest(k, neighbor_states.values(), key=lambda node: node.path_cost)
        current_state_set = list(k_best_nodes)

        if not current_state_set:
            return None
    
    return None   

In [34]:
start =  [[0, 1, 'X'],
          [0, 1, 0],
          [1, 2, 2]]

goal = [[0, 0, 0],
        [0, 0, 0],
        [0, 2, 2]]

result = local_beam_search(start, goal, 3)

print_solution(result)

Bước 0: Khởi tạo
0 1 X
0 1 0
1 2 2

Bước 1: Left và dọn rác ô [0][1]
0 X 0
0 1 0
1 2 2

Bước 2: Down và dọn rác ô [1][1]
0 0 0
0 X 0
1 2 2

Bước 3: Left
0 0 0
X 0 0
1 2 2

Bước 4: Down và dọn rác ô [2][0]
0 0 0
0 0 0
X 2 2

Phòng đã được dọn dẹp!!!


## Simulated Annealing

In [29]:
import random
import math

def simulated_annealing(initial, goal):
    current_node = Node(initial, None, None, calculate_heuristic_cost(initial))

    T = 100      # Gán cố định
    T_min = 0.01
    alpha = 0.99

    while T > T_min: 
        if compare_state(current_node.state, goal):
            return get_solution(current_node)
        
        action, next_state = random.choice(gen_actions(current_node.state))
        h_cost_next_state = calculate_heuristic_cost(next_state)

        delta = h_cost_next_state - current_node.path_cost
        if delta < 0:
            current_node = Node(next_state, current_node, action, h_cost_next_state)
        else:
            p = math.exp(-delta/T)
            if random.uniform(0, 1) < p:
                current_node = Node(next_state, current_node, action, h_cost_next_state)
        
        T *= alpha

    return None

In [31]:
start =  [[0, 1, 'X'],
          [0, 1, 0],
          [1, 2, 2]]

goal = [[0, 0, 0],
        [0, 0, 0],
        [0, 2, 2]]

result = simulated_annealing(start, goal)

print_solution(result)

Không tìm thấy đường đi!!!
